In [ ]:
import os
import numpy as np
from osgeo import gdal
import gc

In [ ]:
def classify_vulnerability_rasters(raster_files, output_dir, nodata_value=-9999, use_alignment_function=False, align_function=None, reference_raster=None):
    """
    Creates vulnerability classification rasters from a list of input rasters.

    Categories:
        LowVulnerability: < 33
        MediumVulnerability: 34–66
        HighVulnerability: 67–100
        UnknownVulnerability: nodata

    Output naming:
        originalname_LowVulnerability.tif
        originalname_MediumVulnerability.tif
        originalname_HighVulnerability.tif
        originalname_UnknownVulnerability.tif
    """

    os.makedirs(output_dir, exist_ok=True)

    for r in raster_files:
        print(f"Processing: {os.path.basename(r)}")

        # ------------------------------------------------------------------
        # Optional alignment step
        # ------------------------------------------------------------------
        if use_alignment_function and align_function and reference_raster:
            r_aligned = align_function(r, reference_raster)
            ds = gdal.Open(r_aligned)
            tmp_to_delete = r_aligned
        else:
            ds = gdal.Open(r)
            tmp_to_delete = None

        if ds is None:
            print(f"ERROR: Cannot open {r}")
            continue

        band = ds.GetRasterBand(1)
        arr = band.ReadAsArray().astype(np.float32)

        if band.GetNoDataValue() is not None:
            nd = band.GetNoDataValue()
        else:
            nd = nodata_value

        # ------------------------------------------------------------------
        # Create category masks
        # ------------------------------------------------------------------
        low_mask     = (arr < 33) & (arr != nd)
        med_mask     = (arr >= 34) & (arr <= 66) & (arr != nd)
        high_mask    = (arr >= 67) & (arr <= 100) & (arr != nd)
        unknown_mask = (arr == nd)

        # ------------------------------------------------------------------
        # Prepare output arrays
        # ------------------------------------------------------------------
        low_arr     = np.where(low_mask, arr, nd)
        med_arr     = np.where(med_mask, arr, nd)
        high_arr    = np.where(high_mask, arr, nd)
        unknown_arr = np.where(unknown_mask, nd, nd)

        # ------------------------------------------------------------------
        # Save function
        # ------------------------------------------------------------------
        def save_output(array, suffix):
            out_path = os.path.join(
                output_dir,
                f"{os.path.splitext(os.path.basename(r))[0]}_{suffix}.tif"
            )
            driver = gdal.GetDriverByName("GTiff")
            out_ds = driver.Create(
                out_path,
                ds.RasterXSize,
                ds.RasterYSize,
                1,
                gdal.GDT_Float32,
                options=["COMPRESS=DEFLATE", "TILED=YES"]
            )
            out_ds.SetGeoTransform(ds.GetGeoTransform())
            out_ds.SetProjection(ds.GetProjection())
            out_ds.GetRasterBand(1).WriteArray(array)
            out_ds.GetRasterBand(1).SetNoDataValue(nd)
            out_ds = None
            print(f"  ✓ Saved: {os.path.basename(out_path)}")

        # ------------------------------------------------------------------
        # Write outputs
        # ------------------------------------------------------------------
        save_output(low_arr,     "LowVulnerability")
        save_output(med_arr,     "MediumVulnerability")
        save_output(high_arr,    "HighVulnerability")
        save_output(unknown_arr, "UnknownVulnerability")

        # ------------------------------------------------------------------
        # Cleanup
        # ------------------------------------------------------------------
        band = None
        ds = None
        del arr, low_arr, med_arr, high_arr, unknown_arr
        gc.collect()

        if tmp_to_delete and os.path.exists(tmp_to_delete):
            try:
                os.remove(tmp_to_delete)
            except PermissionError:
                print(f"Temp file still in use, cannot delete: {tmp_to_delete}")

    print("\nAll vulnerability rasters created successfully.")
    

def get_raster_file_list(path):
    """Get a list of the raster files inside the folder"""
    File_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
    for file in os.listdir(path):
        if file.endswith(".tif") or file.endswith(".tiff"):
            if file not in File_list:
                File_list.append(os.path.join(path,file))
        else:
            pass
    return File_list


In [ ]:
raster_files_path = r"Z:\z_resources\justus\im_nca_postprocessing_ungbf\01_mask_rasters\02_current_processing"
raster_files_list = get_raster_file_list(raster_files_path)

output_dir = r"Z:\z_resources\justus\im_nca_postprocessing_ungbf\03_masked_outputs\vulnerability_files"

In [ ]:
classify_vulnerability_rasters(raster_files_list, output_dir, nodata_value=-9999, use_alignment_function=False, align_function=None, reference_raster=None)